<a href="https://colab.research.google.com/github/deartoms/python/blob/main/day05_practice3_k%EA%B2%B9_%EA%B5%90%EC%B0%A8%EA%B2%80%EC%A6%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 성능 검증 - k겹 교차검증과 드롭아웃
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# 셀 1. 데이터
CSV_URL  = "https://raw.githubusercontent.com/taehojo/deeplearning_4th/master/data/sonar3.csv"
CSV_PATH = "sonar3.csv"
def load_sonar():
  import os
  if os.path.exists(CSV_PATH):
    print(f"로컬 파일 재사용: {CSV_PATH}")
    return pd.read_csv(CSV_PATH, header=None) # 첫 행부터 바로 숫자 데이터라서 header=None 이 필수

  try:
    df = pd.read_csv(CSV_URL, header=None)
    df.to_csv(CSV_PATH, index = False, header=None)
    print(f"다운로드 완료 → {CSV_PATH} 저장 (다음 실행부턴 재사용)")
    return df
  except Exception as e:
    raise SystemExit(
        f"데이터 다운로드 실패: {e}\n"
    )
df = load_sonar()
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
print(f"데이터: {X.shape[0]}개 X 특징 {X.shape[1]}개")
print(f"클래스: 암석 {(y==0).sum()}개 / 금속 {(y==1).sum()}개")

로컬 파일 재사용: sonar3.csv
데이터: 208개 X 특징 60개
클래스: 암석 97개 / 금속 111개


In [5]:
# 셀 2.학습 - 평가 함수
def make_model(dropout=0.0):
  return nn.Sequential(
      nn.Linear(60, 128), nn.ReLU(), nn.Dropout(dropout), #활성화를 통과한 뉴런 출력 중 일부를 끄는 것
      nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
      nn.Linear(64, 1), nn.Sigmoid(),
  )
def train_eval(X_tr, y_tr, X_te, y_te, dropout=0.0, epochs=200):
  scaler = StandardScaler()
  X_tr = scaler.fit_transform(X_tr)
  X_te = scaler.transform(X_te)

  X_tr = torch.tensor(X_tr, dtype=torch.float32).to(device)
  X_te = torch.tensor(X_te, dtype=torch.float32).reshape(-1, 1).to(device)
  y_tr = torch.tensor(y_tr, dtype=torch.float32).to(device)
  y_te = torch.tensor(y_te, dtype=torch.float32).reshape(-1, 1).to(device)

  model = make_model(dropout).to(device)
  loss_fn = nn.BCELoss()
  opt = torch.optim.Adam(model.parameters(), lr=0.005)

  model.train() # 드롭아웃 ON
  for _ in range(epochs):
    loss = loss_fn(model(X_tr), y_tr)
    opt.zero_grad(); loss.backward(); opt.step()

  model.eval() # 드롭아웃 OFF
  with torch.no_grad():
      train_acc = ((model(X_tr) > 0.5) == y_tr.bool()).float().mean().item()
      test_acc = ((model(X_te) > 0.5) == y_te.bool()).float().mean().item()
  return train_acc, test_acc